1. Imports & configuration
2. Load processed datasets
3. Load locked experiment split
4. Verify split integrity
5. Prepare X / y for each model
6. Train 4 Isolation Forest models
7. Validation predictions
8. Select F1 thresholds
9. Freeze models + thresholds
10. Final-test predictions
11. Calculate evaluation metrics
12. Compare Models 1–4
13. Confusion matrices / diagnostic plots
14. Save models, thresholds and results

In [24]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)


# =========================================================
# Configuration
# =========================================================

RANDOM_STATE = 42
N_ESTIMATORS = 300

MAX_MISSING_FRACTION = 0.95


# =========================================================
# Dataset paths
# =========================================================

OPERATION_DATA_PATH = (
    "../data/processed/operation_data.parquet"
)

BASELINE_RAW_WINDOWS_PATH = (
    "../data/processed/baseline_raw_windows.parquet"
)

WINDOW_FEATURES_PATH = (
    "../data/processed/window_features.parquet"
)

TEMPORAL_FEATURES_PATH = (
    "../data/processed/temporal_features.parquet"
)

ML_FEATURES_PATH = (
    "../data/processed/ml_features.parquet"
)


# =========================================================
# Load datasets
# =========================================================

operation_data = pd.read_parquet(
    OPERATION_DATA_PATH
)

baseline_raw_windows = pd.read_parquet(
    BASELINE_RAW_WINDOWS_PATH
)

window_features = pd.read_parquet(
    WINDOW_FEATURES_PATH
)

temporal_features = pd.read_parquet(
    TEMPORAL_FEATURES_PATH
)

ml_features = pd.read_parquet(
    ML_FEATURES_PATH
)


# =========================================================
# Basic inspection
# =========================================================

datasets = {
    "baseline_raw_windows": baseline_raw_windows,
    "window_features": window_features,
    "temporal_features": temporal_features,
    "ml_features": ml_features,
}

for name, df in datasets.items():
    print(
        f"{name:25s} "
        f"shape={df.shape} | "
        f"experiments={df['identifier'].nunique()}"
    )


baseline_raw_windows      shape=(11512, 1810) | experiments=119
window_features           shape=(11512, 250) | experiments=119
temporal_features         shape=(11512, 730) | experiments=119
ml_features               shape=(11512, 787) | experiments=119


In [25]:
# =========================================================
# Step 3 - Load locked experiment split
# =========================================================

EXPERIMENT_SPLIT_PATH = (
    "../data/processed/experiment_split.parquet"
)

experiment_split = pd.read_parquet(
    EXPERIMENT_SPLIT_PATH
)

print("Shape:", experiment_split.shape)
print("\nColumns:")
print(experiment_split.columns.tolist())

print("\nSplit counts:")
display(
    experiment_split["split"]
    .value_counts()
    .sort_index()
)

print("\nLocked experiment split:")
display(
    experiment_split.sort_values(
        ["split", "batch", "operating_point", "experiment"]
    ).reset_index(drop=True)
)

Shape: (79, 7)

Columns:
['identifier', 'batch', 'operating_point', 'experiment', 'experiment_type', 'experiment_class', 'split']

Split counts:


split
final_test    40
validation    39
Name: count, dtype: int64


Locked experiment split:


,identifier,batch,operating_point,experiment,experiment_type,experiment_class,split
0,Operation/batch_dist_ternary_acetone+butan-1-o...,batch_dist_ternary_acetone+butan-1-ol+methanol,operating_point_003,experiment_001,test_anormal,observable_anomaly,final_test
1,Operation/batch_dist_ternary_acetone+butan-1-o...,batch_dist_ternary_acetone+butan-1-ol+methanol,operating_point_005,experiment_001,test_anormal,observable_anomaly,final_test
2,Operation/batch_dist_ternary_acetone+butan-1-o...,batch_dist_ternary_acetone+butan-1-ol+methanol,operating_point_005,experiment_002,test_anormal,observable_anomaly,final_test
3,Operation/batch_dist_ternary_acetone+butan-1-o...,batch_dist_ternary_acetone+butan-1-ol+methanol,operating_point_006,experiment_001,test_anormal,observable_anomaly,final_test
4,Operation/batch_dist_ternary_acetone+butan-1-o...,batch_dist_ternary_acetone+butan-1-ol+methanol,operating_point_008,experiment_001,test_anormal,observable_anomaly,final_test
...,...,...,...,...,...,...,...
74,Operation/batch_dist_ternary_butan-1-ol+propan...,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_026,experiment_001,test_anormal,observable_anomaly,validation
75,Operation/batch_dist_ternary_butan-1-ol+propan...,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_026,experiment_002,test_anormal,observable_anomaly,validation
76,Operation/batch_dist_ternary_butan-1-ol+propan...,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_027,experiment_001,test_anormal,observable_anomaly,validation
77,Operation/batch_dist_ternary_butan-1-ol+propan...,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_030,experiment_001,test_anormal,observable_anomaly,validation


In [26]:
# =========================================================
# Step 4 - Assign train / validation / final_test
# =========================================================

def assign_model_split(df, experiment_split):
    """
    Assign the locked modeling split to a feature dataset.

    train:
        all train_normal experiments

    validation:
        identifiers listed in experiment_split

    final_test:
        identifiers listed in experiment_split
    """

    df = df.copy()

    # -----------------------------------------------------
    # Sanity check: split identifiers must be unique
    # -----------------------------------------------------

    if experiment_split["identifier"].duplicated().any():
        raise ValueError(
            "Duplicate identifiers found in experiment_split."
        )

    # -----------------------------------------------------
    # Start with an empty split column
    # -----------------------------------------------------

    df["model_split"] = pd.Series(
        pd.NA,
        index=df.index,
        dtype="string",
    )

    # -----------------------------------------------------
    # Assign training experiments
    # -----------------------------------------------------

    train_mask = (
        df["experiment_type"] == "train_normal"
    )

    df.loc[
        train_mask,
        "model_split"
    ] = "train"

    # -----------------------------------------------------
    # Map locked validation / final-test assignments
    # -----------------------------------------------------

    split_mapping = (
        experiment_split
        .set_index("identifier")["split"]
    )

    mapped_split = df["identifier"].map(
        split_mapping
    )

    mapped_mask = mapped_split.notna()

    df.loc[
        mapped_mask,
        "model_split"
    ] = mapped_split.loc[mapped_mask].astype("string")

    return df


# =========================================================
# Apply the locked split to all four datasets
# =========================================================

baseline_raw_windows = assign_model_split(
    baseline_raw_windows,
    experiment_split,
)

window_features = assign_model_split(
    window_features,
    experiment_split,
)

temporal_features = assign_model_split(
    temporal_features,
    experiment_split,
)

ml_features = assign_model_split(
    ml_features,
    experiment_split,
)


# =========================================================
# Verify split assignments
# =========================================================

datasets = {
    "baseline_raw_windows": baseline_raw_windows,
    "window_features": window_features,
    "temporal_features": temporal_features,
    "ml_features": ml_features,
}

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * len(name))

    experiment_split_check = (
        df[
            [
                "identifier",
                "model_split",
            ]
        ]
        .drop_duplicates()
        ["model_split"]
        .value_counts()
        .sort_index()
    )

    display(
        experiment_split_check
        .rename("n_experiments")
        .to_frame()
    )


baseline_raw_windows
--------------------


,n_experiments
model_split,
final_test,40
train,38
validation,39



window_features
---------------


,n_experiments
model_split,
final_test,40
train,38
validation,39



temporal_features
-----------------


,n_experiments
model_split,
final_test,40
train,38
validation,39



ml_features
-----------


,n_experiments
model_split,
final_test,40
train,38
validation,39


In [27]:
# =========================================================
# Step 5 — Verify no experiment leakage
# =========================================================

for name, df in datasets.items():

    experiment_splits = (
        df[
            [
                "identifier",
                "model_split",
            ]
        ]
        .drop_duplicates()
    )

    # Count how many different splits each identifier has
    split_counts = (
        experiment_splits
        .groupby("identifier")["model_split"]
        .nunique()
    )

    leaked_identifiers = split_counts[
        split_counts > 1
    ]

    print(f"\n{name}")
    print("-" * len(name))

    if len(leaked_identifiers) == 0:
        print("✓ No experiment appears in multiple splits.")
    else:
        print("✗ LEAKAGE DETECTED:")
        display(leaked_identifiers)


# =========================================================
# Verify the split file itself
# =========================================================

print("\nLocked split file:")
print(
    experiment_split.groupby("split")["identifier"]
    .nunique()
    .sort_index()
)

print("\n✓ Split verification complete.")


baseline_raw_windows
--------------------
✓ No experiment appears in multiple splits.

window_features
---------------
✓ No experiment appears in multiple splits.

temporal_features
-----------------
✓ No experiment appears in multiple splits.

ml_features
-----------
✓ No experiment appears in multiple splits.

Locked split file:
split
final_test    40
validation    39
Name: identifier, dtype: int64

✓ Split verification complete.


In [28]:
# =========================================================
# Step 6 — Create model-specific train / validation / test
# =========================================================

def create_model_datasets(df):
    """
    Split a feature dataset according to the locked
    experiment-level split.

    Train:
        38 normal experiments

    Validation:
        validation anomalous experiments + normal experiments

    Final test:
        final-test anomalous experiments + normal experiments
    """

    train = df[
        df["model_split"] == "train"
    ].copy()

    validation_anomalies = df[
        df["model_split"] == "validation"
    ].copy()

    final_test_anomalies = df[
        df["model_split"] == "final_test"
    ].copy()

    return (
        train,
        validation_anomalies,
        final_test_anomalies,
    )


# =========================================================
# Apply to all four datasets
# =========================================================

model_datasets = {}

for name, df in datasets.items():

    (
        train_df,
        validation_df,
        final_test_df,
    ) = create_model_datasets(df)

    model_datasets[name] = {
        "train": train_df,
        "validation": validation_df,
        "final_test": final_test_df,
    }


# =========================================================
# Display experiment counts
# =========================================================

summary_rows = []

for name, splits in model_datasets.items():

    summary_rows.append({
        "model_dataset": name,
        "train_experiments": splits["train"]["identifier"].nunique(),
        "validation_experiments": splits["validation"]["identifier"].nunique(),
        "final_test_experiments": splits["final_test"]["identifier"].nunique(),
    })

model_dataset_summary = pd.DataFrame(
    summary_rows
)

display(model_dataset_summary)

,model_dataset,train_experiments,validation_experiments,final_test_experiments
0,baseline_raw_windows,38,39,40
1,window_features,38,39,40
2,temporal_features,38,39,40
3,ml_features,38,39,40


In [29]:
def add_observable_anomaly_target(df):
    """
    Create the target for observable anomaly detection.

    anomaly_label:
        0 = normal
        1 = blind phase -> exclude
        2 = detectable anomaly -> anomaly
        3 = after-effects still observable -> anomaly

    observable_anomaly:
        0 = normal
        1 = observable anomaly
        NaN = blind phase / excluded
    """
    df = df.copy()

    if "anomaly_label" not in df.columns:
        raise ValueError(
            "'anomaly_label' column not found."
        )

    df["observable_anomaly"] = np.nan

    # Normal
    df.loc[
        df["anomaly_label"] == 0,
        "observable_anomaly"
    ] = 0

    # Detectable anomaly states
    df.loc[
        df["anomaly_label"].isin([2, 3]),
        "observable_anomaly"
    ] = 1

    # Label 1 remains NaN and is excluded
    return df


# =========================================================
# Add target to all four model datasets
# =========================================================

baseline_raw_windows = add_observable_anomaly_target(
    baseline_raw_windows
)

window_features = add_observable_anomaly_target(
    window_features
)

temporal_features = add_observable_anomaly_target(
    temporal_features
)

ml_features = add_observable_anomaly_target(
    ml_features
)


# =========================================================
# Dataset collection
# =========================================================

datasets = {
    "baseline_raw_windows": baseline_raw_windows,
    "window_features": window_features,
    "temporal_features": temporal_features,
    "ml_features": ml_features,
}


# =========================================================
# Check target distribution
# in validation and final test
# =========================================================

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * len(name))

    evaluation_data = df[
        df["model_split"].isin(
            ["validation", "final_test"]
        )
        & df["observable_anomaly"].notna()
    ].copy()

    target_distribution = (
        evaluation_data
        .groupby("model_split")["observable_anomaly"]
        .agg(
            n_windows="size",
            n_anomalies="sum",
            anomaly_rate="mean",
        )
    )

    display(target_distribution)


baseline_raw_windows
--------------------


,n_windows,n_anomalies,anomaly_rate
model_split,,,
final_test,3839,1168.0,0.304246
validation,3531,1071.0,0.303314



window_features
---------------


,n_windows,n_anomalies,anomaly_rate
model_split,,,
final_test,3839,1168.0,0.304246
validation,3531,1071.0,0.303314



temporal_features
-----------------


,n_windows,n_anomalies,anomaly_rate
model_split,,,
final_test,3839,1168.0,0.304246
validation,3531,1071.0,0.303314



ml_features
-----------


,n_windows,n_anomalies,anomaly_rate
model_split,,,
final_test,3839,1168.0,0.304246
validation,3531,1071.0,0.303314


In [30]:
# ============================================================
# Verify model inputs before training
# ============================================================

EXPECTED_FEATURE_COUNTS = {
    "baseline_raw_windows": 1800,
    "window_features": 240,
    "temporal_features": 720,
    "ml_features": 780,
}

datasets = {
    "baseline_raw_windows": baseline_raw_windows,
    "window_features": window_features,
    "temporal_features": temporal_features,
    "ml_features": ml_features,
}

print("=" * 70)
print("MODEL INPUT VALIDATION")
print("=" * 70)

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * len(name))

    # --------------------------------------------------------
    # Identify ML feature columns
    # --------------------------------------------------------

    metadata_columns = [
        "identifier",
        "batch",
        "operating_point",
        "experiment",
        "experiment_type",
        "window",
        "window_start",
        "window_end",
        "anomaly_label",
        "observable_anomaly",
        "model_split",
    ]

    feature_columns = [
        column
        for column in df.columns
        if column not in metadata_columns
    ]

    # --------------------------------------------------------
    # Basic dimensions
    # --------------------------------------------------------

    print(f"Shape:              {df.shape}")
    print(f"ML features:        {len(feature_columns)}")
    print(
        f"Expected features:  "
        f"{EXPECTED_FEATURE_COUNTS[name]}"
    )

    # --------------------------------------------------------
    # Feature count check
    # --------------------------------------------------------

    if len(feature_columns) != EXPECTED_FEATURE_COUNTS[name]:
        raise ValueError(
            f"{name}: unexpected feature count."
        )

    # --------------------------------------------------------
    # Split counts
    # --------------------------------------------------------

    split_counts = (
        df["model_split"]
        .value_counts(dropna=False)
        .sort_index()
    )

    print("\nWindows per split:")
    print(split_counts)

    # --------------------------------------------------------
    # Missing values in ML features
    # --------------------------------------------------------

    missing_features = (
        df[feature_columns]
        .isna()
        .sum()
    )

    n_features_with_missing = (
        (missing_features > 0)
        .sum()
    )

    n_missing_values = (
        missing_features
        .sum()
    )

    print(
        f"\nFeatures containing NaN: "
        f"{n_features_with_missing}"
    )

    print(
        f"Total NaN values:        "
        f"{n_missing_values:,}"
    )

    # --------------------------------------------------------
    # Infinite values
    # --------------------------------------------------------

    infinite_values = (
        np.isinf(
            df[feature_columns]
            .select_dtypes(include=np.number)
        )
        .sum()
        .sum()
    )

    print(
        f"Infinite values:         "
        f"{infinite_values:,}"
    )

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)


MODEL INPUT VALIDATION

baseline_raw_windows
--------------------
Shape:              (11512, 1811)
ML features:        1800
Expected features:  1800

Windows per split:
model_split
final_test    3932
train         3738
validation    3716
<NA>           126
Name: count, dtype: Int64

Features containing NaN: 0
Total NaN values:        0
Infinite values:         0

window_features
---------------
Shape:              (11512, 252)
ML features:        241
Expected features:  240


ValueError: window_features: unexpected feature count.